<a href="https://colab.research.google.com/github/HariKarthikeyaManiShankarReddy/Document-Optical-Charcter-Recognition-/blob/main/Optical_Character_Recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# importing required libraries for the project
!pip install numpy==1.26.4
!pip install opencv-python==4.10.0.84
!pip install paddlepaddle==3.2.2
!pip install paddleocr

  Using cached opencv_python-4.10.0.84-cp37-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (20 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 MB 10.7 MB/s eta 0:00:00
  Attempting uninstall: opencv-python
    Found existing installation: opencv-python 4.13.0.92
    Uninstalling opencv-python-4.13.0.92:
      Successfully uninstalled opencv-python-4.13.0.92
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 189.3/189.3 MB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.6 MB/s eta 0:00:00
  Attempting uninstall: opt_einsum
    Found existing installation: opt_einsum 3.4.0
    Uninstalling opt_einsum-3.4.0:
      Successfully uninstalled opt_einsum-3.4.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
# Initializing the Optical Charcter Recognition Engine for the project
import os
os.environ.setdefault("FLAGS_enable_pir_api", "0")
# flags_enable_pir_api used to disable the new paddle intermediate representation system
from paddleocr import PaddleOCR

ocr = PaddleOCR(
    lang="en",
    use_textline_orientation=True,
    enable_mkldnn=False,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
)

/usr/local/lib/python3.12/dist-packages/paddle/utils/cpp_extension/extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_textline_ori', None, None)
Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.
Using official model (PP-LCNet_x1_0_textline_ori), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-LCNet_x1_0_textline_ori`.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Goog

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

Creating model: ('PP-OCRv6_medium_det', None, None)
Using official model (PP-OCRv6_medium_det), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_medium_det`.


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Creating model: ('PP-OCRv6_medium_rec', None, None)
Using official model (PP-OCRv6_medium_rec), the model files will be automatically downloaded and saved in `/root/.paddlex/official_models/PP-OCRv6_medium_rec`.


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

In [ ]:
# converting the image to gray scale image and preprocessing
import cv2

def preprocess_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise FileNotFoundError(f"Could not read image: {image_path}")
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    threshold = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY, 11, 2
    )
    # cv2.thresh_bianry is used to convert grayscale images into binary images
    # cv2.adaptive_thresh_gaussian_c is used to convert grayscale image into high contrast binary images.
    return threshold

In [ ]:
# Extracting text from the image
def extract_text(image_path):
    processed = preprocess_image(image_path)
    temp_file = "temp_receipt.png"
    cv2.imwrite(temp_file, processed)
    results = ocr.predict(temp_file)
    text_lines = []
    for result in results:
        if hasattr(result, "json"):
            res = result.json.get("res", {})
        elif isinstance(result, dict):
            res = result.get("res", result)
        else:
            res = {}
        for text in res.get("rec_texts", []):
            if text:
                text_lines.append(str(text))
    return "\n".join(text_lines)

In [ ]:
import re

NON_ITEM_PATTERN = re.compile(
    r"\b(total|subtotal|tax|gst|vat|cgst|sgst|change|cash|balance|"
    r"amount due|grand total|net amount|paid|discount|savings|tip)\b",
    re.IGNORECASE,
)
PRICE_ONLY_PATTERN = re.compile(r"^(?:[\$₹]|Rs\.?\s*)?\d+\.\d{2}$", re.IGNORECASE)
REFERENCE_PATTERN = re.compile(
    r"(Bill|Invoice|Order)\s*(?:No|Number|Num|#)?\s*[:\-]?\s*([A-Za-z0-9\-]+)",
    re.IGNORECASE,
)


def clean_item_name(name):
    name = re.sub(r"\s+", " ", name).strip()
    name = re.sub(r"^\d+\s*x\s*", "", name, flags=re.IGNORECASE)
    name = re.sub(r"\s*x\s*\d+\s*$", "", name, flags=re.IGNORECASE)
    return name.strip(" .-:")


def parse_item_name(line):
    """Return item name only (no price) from one OCR line."""
    line = line.strip()
    if not line or NON_ITEM_PATTERN.search(line) or PRICE_ONLY_PATTERN.match(line):
        return None

    if re.search(r"\d{2}[-/]\d{2}[-/]\d{4}", line):
        return None
    if re.fullmatch(r"\d{2}:\d{2}(:\d{2})?", line):
        return None
    if REFERENCE_PATTERN.search(line):
        return None

    prices = re.findall(r"\d+\.\d{2}", line)
    if prices:
        name_part = line[: line.rfind(prices[-1])].strip()
        name_part = re.sub(r"(?:[\$₹]|Rs\.?\s*)$", "", name_part, flags=re.IGNORECASE).strip()
    else:
        name_part = line

    name = clean_item_name(name_part)
    if name and re.search(r"[A-Za-z]{2,}", name):
        return name
    return None


def extract_items(text):
    """Return a list of item names only."""
    items = []

    for raw_line in text.split("\n"):
        name = parse_item_name(raw_line)
        if name and name not in items:
            items.append(name)

    return items


def extract_reference_number(text):
    """Find Bill No / Invoice No / Order No."""
    match = REFERENCE_PATTERN.search(text)
    if match:
        return match.group(2)

    # OCR sometimes splits label and number across two lines
    lines = [line.strip() for line in text.split("\n") if line.strip()]
    for i, line in enumerate(lines):
        if re.search(r"^(Bill|Invoice|Order)\s*(No|Number|Num|#)?\s*[:\-]?$", line, re.IGNORECASE):
            if i + 1 < len(lines):
                next_line = lines[i + 1]
                if re.fullmatch(r"[A-Za-z0-9\-]+", next_line):
                    return next_line
    return "Not Found"


def extract_total_amount(text):
    """Find total / grand total / net amount."""
    patterns = [
        r"(?:grand\s*)?total\s*(?:amount)?\s*[:\-]?\s*(?:[\$₹]|Rs\.?\s*)?(\d+\.\d{2})",
        r"net\s*amount\s*[:\-]?\s*(?:[\$₹]|Rs\.?\s*)?(\d+\.\d{2})",
        r"amount\s*due\s*[:\-]?\s*(?:[\$₹]|Rs\.?\s*)?(\d+\.\d{2})",
    ]
    for pattern in patterns:
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            return float(match.group(1))
    return "Not Found"


def extract_details(text):
    data = {
        "date": "Not Found",
        "time": "Not Found",
        "bill_number": "Not Found",
        "items": [],
        "total_amount": "Not Found",
    }

    date_match = re.search(r"\d{2}[-/]\d{2}[-/]\d{4}", text)
    if date_match:
        data["date"] = date_match.group()

    time_match = re.search(r"\d{2}:\d{2}(:\d{2})?", text)
    if time_match:
        data["time"] = time_match.group()

    data["bill_number"] = extract_reference_number(text)
    data["items"] = extract_items(text)
    data["total_amount"] = extract_total_amount(text)

    return data

In [ ]:
# Preprocessing the receipt image and saving the results
import json

def process_receipt(image_path):
    text = extract_text(image_path)
    details = extract_details(text)

    print("\n===== RECEIPT DETAILS =====")
    print("Date          :", details["date"])
    print("Time          :", details["time"])
    print("Bill/Order No :", details["bill_number"])
    print("Total Amount  :", details["total_amount"])
    print("\nItems:")
    if details["items"]:
        for item in details["items"]:
            print("-", item)
    else:
        print("- No items found")

    with open("receipt_output.json", "w") as file:
        json.dump(details, file, indent=4)

    print("\nResults saved to receipt_output.json")

In [ ]:
image_path="sample2.jpg"
process_receipt(image_path)


===== RECEIPT DETAILS =====
Date          : 19/10/2018
Time          : 20:49:59
Bill/Order No : Not Found
Total Amount  : Not Found

Items:
- tan woon yann
- INDAH GIFT & HOME DECO
- 27,JALAN DEDAP 13
- TAMAN JOHOR JAYA,
- 81100 JOHOR BAHRU,JOHOR
- Tel:07-3507405 Fax:07-3558160
- RECEIPT
- Cashier: CH
- Location/SP: 05 /0531
- MB: M026588
- Room No: 01
- Desc/Iten
- Qty Price: Amt(RM)
- ST-PRIVILEGE CARD/GD INDAH
- GF-TABLE LAMP/STITCH <i>
- eDISC
- TOTALAHT
- RM
- ROUNDING ADJ
- RH
- RN
- Thank You ! Please Come Again
- Goods Sold Are Not Returnable
- Dealing In Wholesale, And Retail

Results saved to receipt_output.json
